# 03 — Faz 3b: OOF Propensity Ablation

**Framing (kilitli):** cross-sectional propensity modeli. Tek snapshot (Ekim 2023),
zamansal tahmin değil; SHAP ilişkisel.

**Soru:** Ürün-profili baseline'ına (3a leak-clean) OOF seller/category propensity
eklemek **dürüst** bir lift sağlıyor mu?

**Karar kuralı (önceden taahhüt):** Bir kol ancak şu iki şart birlikte sağlanırsa
"kazandı" sayılır:
1. **Hold-out** PR-AUC, bir önceki koldan ≥ 0.005 yüksek (CV değil, hold-out).
2. **train−CV gap genişlemiyor** (≤ +0.01 tolerans). Gap genişliyorsa model
   grup kimliğini ezberliyordur → CV artsa bile lift sayılmaz.

**Kontrol:** Üç kol da **aynı hafif-regularize LightGBM** config'iyle koşulur ki
delta'lar genelleme kapasitesini yansıtsın (3a'nın 0.142 gap'li ezberci GBM'i değil).

OOF mekaniği: `TargetEncoder` (cross-fitted) bir **Pipeline adımı** — her CV fold'unda
yalnız o fold'un train'inde fit edilir, dış test fold'unu görmez.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import json
import numpy as np, pandas as pd, joblib
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, StandardScaler, TargetEncoder
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.metrics import average_precision_score, roc_auc_score
from lightgbm import LGBMClassifier
from rebuild import rebuild, TARGET

SEED = 42
np.random.seed(SEED)
print("setup ok")

setup ok


In [2]:
# Leak-clean Option-A feature setleri (3a ile ayni) + OOF icin ham gruplar
ROBUST   = ["variation_count", "offer_count_trend", "Package: Weight (g)"]
STANDARD = ["Reviews: Rating", "new_price_log", "review_count_log", "sr_log",
            "review_velocity", "new_price_margin_est", "sr_drops_90",
            "Package: Dimension (cm³)"]
PASS     = ["has_sales_data", "is_negative_margin", "product_age_segment", "is_active_seller"]
FREQ     = ["Categories: Sub", "Brand"]          # frequency-encode (hedeften bagimsiz)
TGT_CAT  = ["Categories: Root"]                  # OOF target-encode (kategori propensity)
TGT_SEL  = ["listing_seller"]                    # OOF target-encode (satici propensity)

rb = rebuild()
y  = rb[TARGET].astype(int)
BASE = float(y.mean())
X = rb[ROBUST + STANDARD + PASS + FREQ + TGT_CAT + TGT_SEL].copy()

# 3a ile ayni kilitli %15 hold-out
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.15, stratify=y, random_state=SEED)
print(f"base rate={BASE:.4f} | train={len(X_tr):,} | holdout={len(X_te):,}")

base rate=0.1457 | train=46,626 | holdout=8,229


In [3]:
class FrequencyEncoder(BaseEstimator, TransformerMixin):
    """Frequency encoding, min_count rare->'other'. Train'de fit."""
    def __init__(self, min_count=50): self.min_count = min_count
    def fit(self, X, y=None):
        X = pd.DataFrame(X); self.maps_, self.rare_ = {}, {}
        for c in X.columns:
            f = X[c].value_counts(); rare = set(f[f < self.min_count].index)
            coll = X[c].where(~X[c].isin(rare), "other")
            self.maps_[c] = coll.value_counts(normalize=True).to_dict(); self.rare_[c] = rare
        return self
    def transform(self, X):
        X = pd.DataFrame(X); out = np.zeros((len(X), X.shape[1]))
        for j, c in enumerate(X.columns):
            coll = X[c].where(~X[c].isin(self.rare_[c]), "other")
            out[:, j] = coll.map(self.maps_[c]).fillna(0).values
        return out
print("FrequencyEncoder ready")

FrequencyEncoder ready


In [4]:
def make_pipe(target_cols):
    """target_cols: OOF target-encode edilecek ham grup kolonlari (bos olabilir)."""
    steps = [
        ("robust",   RobustScaler(),            ROBUST),
        ("standard", StandardScaler(),          STANDARD),
        ("freq",     FrequencyEncoder(50),      FREQ),
        ("pass",     "passthrough",             PASS),
    ]
    if target_cols:
        # cross-fitted: fold-ici leakage yok; bilinmeyen grup -> prior (global mean) = fallback
        steps.append(("tgt", TargetEncoder(target_type="binary", cv=5, random_state=SEED), target_cols))
    pre = ColumnTransformer(steps, remainder="drop")
    spw = float((y_tr == 0).sum() / (y_tr == 1).sum())
    clf = LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=15,
                         min_child_samples=100, subsample=0.8, colsample_bytree=0.8,
                         reg_lambda=5.0, scale_pos_weight=spw,
                         random_state=SEED, n_jobs=-1, verbose=-1)
    return Pipeline([("pre", pre), ("clf", clf)])

ARMS = {"A_urun_only": [], "B_+category": TGT_CAT, "C_+category+seller": TGT_CAT + TGT_SEL}
print("arms:", list(ARMS))

arms: ['A_urun_only', 'B_+category', 'C_+category+seller']


In [5]:
skf = StratifiedKFold(5, shuffle=True, random_state=SEED)
rows = []
for name, tcols in ARMS.items():
    pipe = make_pipe(tcols)
    r = cross_validate(pipe, X_tr, y_tr, cv=skf,
                       scoring=["average_precision", "roc_auc"],
                       return_train_score=True, n_jobs=-1)
    cv_pr, tr_pr = r["test_average_precision"].mean(), r["train_average_precision"].mean()
    # dokunulmamis hold-out
    p = make_pipe(tcols).fit(X_tr, y_tr).predict_proba(X_te)[:, 1]
    rows.append({"arm": name,
                 "cv_pr_auc": round(cv_pr, 4),
                 "cv_std": round(r["test_average_precision"].std(), 4),
                 "holdout_pr_auc": round(average_precision_score(y_te, p), 4),
                 "roc_cv": round(r["test_roc_auc"].mean(), 4),
                 "train_cv_gap": round(tr_pr - cv_pr, 4)})
res = pd.DataFrame(rows).set_index("arm")
res

,cv_pr_auc,cv_std,holdout_pr_auc,roc_cv,train_cv_gap
arm,,,,,
A_urun_only,0.6317,0.0292,0.6549,0.9102,0.1038
B_+category,0.6403,0.0293,0.6525,0.9136,0.0994
C_+category+seller,0.6564,0.0338,0.6765,0.9183,0.0956


In [6]:
# Karar kurali: hold-out lift >= 0.005 VE gap genislemesi <= 0.01
EPS, GAP_TOL = 0.005, 0.01
verdicts = {}
order = ["A_urun_only", "B_+category", "C_+category+seller"]
for i in range(1, len(order)):
    cur, prev = order[i], order[i-1]
    ho_lift = res.loc[cur, "holdout_pr_auc"] - res.loc[prev, "holdout_pr_auc"]
    gap_inc = res.loc[cur, "train_cv_gap"] - res.loc[prev, "train_cv_gap"]
    won = (ho_lift >= EPS) and (gap_inc <= GAP_TOL)
    verdicts[cur] = {"holdout_lift": round(ho_lift, 4), "gap_increase": round(gap_inc, 4),
                     "verdict": "KAZANDI" if won else "SAYILMAZ"}
    print(f"{cur:22s} hold-out lift={ho_lift:+.4f}  gap delta={gap_inc:+.4f}  -> {verdicts[cur]['verdict']}")
verdicts

B_+category            hold-out lift=-0.0024  gap delta=-0.0044  -> SAYILMAZ
C_+category+seller     hold-out lift=+0.0240  gap delta=-0.0038  -> KAZANDI


{'B_+category': {'holdout_lift': np.float64(-0.0024),
  'gap_increase': np.float64(-0.0044),
  'verdict': 'SAYILMAZ'},
 'C_+category+seller': {'holdout_lift': np.float64(0.024),
  'gap_increase': np.float64(-0.0038),
  'verdict': 'KAZANDI'}}

In [7]:
# Sonuclari experiment_log.json'a ekle; seller kolu kazandiysa encoder'i (fallback'li) kaydet
log = json.load(open("experiment_log.json"))
log["phase_3b_ablation"] = {
    "config": "lightly-regularized LightGBM (num_leaves=15, min_child_samples=100, reg_lambda=5)",
    "decision_rule": "holdout lift >= 0.005 AND train-CV gap increase <= 0.01",
    "results": res.reset_index().to_dict(orient="records"),
    "verdicts": verdicts,
}
json.dump(log, open("experiment_log.json", "w"), indent=2, ensure_ascii=False)

seller_won = verdicts.get("C_+category+seller", {}).get("verdict") == "KAZANDI"
if seller_won:
    enc = TargetEncoder(target_type="binary", cv=5, random_state=SEED).fit(X_tr[TGT_SEL], y_tr)
    joblib.dump({"encoder": enc, "global_mean_fallback": BASE}, "seller_rate_map.pkl")
    print("seller kolu kazandi -> seller_rate_map.pkl kaydedildi (bilinmeyen satici -> global mean)")
else:
    print("seller kolu SAYILMAZ -> eklenmedi; baseline sadelik korunuyor")
print("experiment_log.json guncellendi")

seller kolu kazandi -> seller_rate_map.pkl kaydedildi (bilinmeyen satici -> global mean)
experiment_log.json guncellendi


## Sonuç

Karar kuralı (hold-out lift ≥ 0.005 **ve** gap genişlemesi ≤ 0.01) yukarıdaki
`verdicts` çıktısında uygulandı. Kazanan kol(lar) feature setine eklenir;
`+seller` kazandıysa serving için `seller_rate_map.pkl` (bilinmeyen satıcıya
global-mean fallback) kaydedildi. Sonraki adım: thin e2e iskelet (model → FastAPI → Docker → lokal).